# Elasticsearchのベクトル検索のシンプルなサンプル

Elasticsearchクライアントを使わず、requestsを使い、REST APIへリクエストすることでElasticsearchを扱う

※Elasticsearchのバージョンアップ後、Elasticsearchクライアントの対応まで時間がかかることがあり、REST APIでの操作を推奨する旨、Elastic社のサポートチームやコミュニティでも議論されているため。

## 必要パッケージのインポート

In [ ]:
import requests
# from elasticsearch8 import Elasticsearch
from sentence_transformers import SentenceTransformer
import json
import time

## 設定

In [ ]:
ES_URL = 'http://llm-rag-examples-elasticsearch1:9200'
INDEX_NAME = 'vector_test01'
HEADERS = {
    'Accept': 'application/vnd.elasticsearch+json; compatible-with=8',
    'Content-Type': 'application/vnd.elasticsearch+json; compatible-with=8',
}

In [ ]:
MODEL_NAME = 'all-MiniLM-L6-v2'
MODEL_DIM = 384

## 埋め込みモデル初期化

In [ ]:
model = SentenceTransformer(MODEL_NAME)

## インデックスの有無チェック、なければエラーで終了

In [ ]:
res = requests.head(f"{ES_URL}/{INDEX_NAME}")
exists_index = res.status_code == 200

if not exists_index:
    raise Exception(f'index {INDEX_NAME} がありません。まず、登録処理を実行してください。（elasticsearch-vector-ex01-01-request-insert.ipynb）')

## ベクトル検索

In [ ]:
QUERY_TEXT = '日本の都市'
TOP_K = 3

In [ ]:
query_vector = model.encode(QUERY_TEXT)
query = {
    'knn': {
        'field': 'vector',
        'query_vector': query_vector.tolist(),
        'k': TOP_K,
        'num_candidates': 100
    }
}
body = {
    'size': TOP_K,
    'query': query
}
res = requests.post(f"{ES_URL}/{INDEX_NAME}/_search", headers=HEADERS, data=json.dumps(body))
res.raise_for_status()

hits = res.json()['hits']['hits']
print("--- 検索結果 ---")
for hit in hits:
    print(f"スコア: {hit['_score']:.4f}, テキスト: {hit['_source']['text']}")